<font color=#009afa size=50>DAT5001 - Visualisation of Data</font>

<font size=50>Satellite Dataset, for PRES1</font>

|**Assessment Code:** |PRES1|
|---|---|
|**Author:**|Tom Rowan, ST20285213|
|**Course:**| DAT5001_S2_25|
|**Course Tutor:**| ***** Dr. Imtiaz *****|

In [105]:
# Install Requirements Using Pip
%pip install -r requirements.txt

Note: you may need to restart the kernel to use updated packages.


In [106]:
# Standard Libraries
import json
import random
from datetime import datetime
import sys

# Pandas and numpy
import pandas as pd
import numpy as np

# Matplot Lib
import matplotlib.pyplot as plt
from matplotlib import colormaps
from matplotlib.pyplot import figure
%matplotlib inline

# Seaborn Graphs
import seaborn as sns

# Cartopy Maps
import cartopy.crs as ccrs
import cartopy.feature as cfeature

# Geopandas
import geopandas as gpd

# GeoPy
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter

# Pycountry 
import pycountry

# Tabulate
from tabulate import tabulate

# Colorama
from colorama import Fore, Back, Style

# Requests
import requests

# Beautifulsoup
from bs4 import BeautifulSoup

# Do not print warnings!
import warnings
warnings.filterwarnings('ignore')

### Set up Colour Palettes for Graphs and Charts

In [107]:
custom_palette = ['#4977AAff','#FFBD02ff','#395B83ff','#989DB1ff','#686347ff','#FFFFFFff','#565E5Eff','#3E4A58ff']
sns.set_palette(sns.color_palette(custom_palette))

### Load the Datasets from the Internet

Reference Website:
https://www.planet4589.org/space/gcat/index.html

Tab Separated Value Files: 

- SatCat: https://www.planet4589.org/space/gcat/tsv/cat/satcat.tsv
- Payload: https://www.planet4589.org/space/gcat/tsv/cat/psatcat.tsv   
- Org Data: https://www.planet4589.org/space/gcat/tsv/tables/orgs.tsv
- Launch sites: https://www.planet4589.org/space/gcat/tsv/tables/sites.tsv

Documentation: https://www.planet4589.org/space/gcat/pdf/doc.pdf

Reference: "McDowell, J, 2020: General Catalog of Artificial Space Objects, https://planet4589.org/space/gcat", First accessed 24/10/2025

> VIII.2.1.1 The Standard (S) Satellite Catalog
>
> Since the late 1950s the US government has maintained a catalog of satellites (often referred to as the SATCAT). Each
satellite is given a sequential number (1 was the rocket stage from the first Sputnik). These numbers are sometimes
referred to as SSN numbers.
> 
> In GCAT, each entry in the S catalog (Standard Satellite Catalog or satcat) corresponds directly to the
corresponding entry in the US catalog; entry S00001 corresponds to SSN 1, entry S40000 corresponds to SSN 40000.
> 
> The S catalog contains additional metadata not present in the US catalog, described later in this chapter. Note that I
will use SATCAT in upper case to refer to the US catalog, while the lower case version will always refer to GCAT’s
S catalog.
> 
> By convention I render entries 1-99999 in each catalog with 5 digits, in this case S00001 to S99999. I then
jump to 9 digits for objects 000100000 onwards (S000100000 in this case). I am optimistic that we won’t reach
S999999999 before I retire, so that’ll be Somebody Else’s Problem.
> 
> There are, as always, complications. Not infrequently, the SSN numbers associated with two space objects
are swapped, or an SSN number is reassigned to a different object and the previously assigned object is now not
in the catalog. However, archival TLE (Two Line Element) sets for the previous object remain associated with the
same catalog number, potentially causing confusion. Although I would rather not reassign numbers, in the interests
of practical compatibility the S catalog entry corresponds to the object MOST RECENTLY ASSOCIATED with that
catalog number (and the GCAT is updated as required to reflect this).
> 
> The SSN number is really associated with an object tracked by the US space tracking team (currently the 18th
Space Control Squadron and associated organizations). The set of TLEs associated with that catalog number, for the
most part, define the SSN. The association of that catalog number with a physical object (satellite, rocket, debris) is
an inference. Sometimes I disagree with the association made, and the S catalog reflects that. The harder problem
is when you have, say, 3 SSN tracked objects and three known payloads, but you have no idea which is which. This
can occur, for example, when three externally similar cubesats deployed from the same vehicle immediately fail, so
even the satellite owners don’t know which is which. In this case the SATCAT will just have them as ’OBJECT C’,
etc., and not attempt to make an identification. I prefer to make an arbitrary assignment of the three objects to catalog
numbers but flag the assignments as uncertain. This lets me enter all the metadata such as country, owner, name etc.
and so statistical counts such as number of payloads associated with a given country will come out right.
> 
> Some satellites are made up of several attached sections. When I deem that these separate sections are of
sufficient interest to deserve their own catalog entry (for example because their metadata are interestingly different)
I create additional entries in the auxcat (see below), and choose one of the sections to be associated with the S entry.
As an example, for CORONA satellites both the CORONA payload and the attached Agena rocket stage, as well as
any Agena aft rack attached payloads, get catalog entries. In such cases the S entry is given to the payload; metadata
includes the information that the CORONA and the Agena remained attached until reentry.
> 
> The SATCAT associates debris objects with a particular launch but does not identify them in detail; again, I
make my best guess as to the nature and origin of particular debris objects.
> 
> The US catalog is coy about certain military and intelligence satellites belonging to the US and its allies.
Usually a cover name (e.g. USA 304) and a launch date are provided, but no orbital data or reentry date. Since I do
not have access to secret information, I am free to speculate. Usually it’s not very hard and identifications of secret
satellites with particular programs and orbits mostly have a high degree of confidence.

In [108]:
# This will get the very latest dataset from John McDowells's page. This dataset is updated daily!
satcat_df = pd.read_csv('https://www.planet4589.org/space/gcat/tsv/cat/satcat.tsv', sep='\t')
pay_df = pd.read_csv('https://www.planet4589.org/space/gcat/tsv/cat/psatcat.tsv', sep='\t')
orgs_df = pd.read_csv('https://www.planet4589.org/space/gcat/tsv/tables/orgs.tsv', sep='\t')
lsite_df = pd.read_csv('https://www.planet4589.org/space/gcat/tsv/tables/sites.tsv', sep='\t')

### Explore these datasets

In [109]:
satcat_df.columns

Index(['#JCAT', 'Satcat', 'Launch_Tag', 'Piece', 'Type', 'Name', 'PLName',
       'LDate', 'Parent', 'SDate', 'Primary', 'DDate', 'Status', 'Dest',
       'Owner', 'State', 'Manufacturer', 'Bus', 'Motor', 'Mass', 'MassFlag',
       'DryMass', 'DryFlag', 'TotMass', 'TotFlag', 'Length', 'LFlag',
       'Diameter', 'DFlag', 'Span', 'SpanFlag', 'Shape', 'ODate', 'Perigee',
       'PF', 'Apogee', 'AF', 'Inc', 'IF', 'OpOrbit', 'OQUAL', 'AltNames'],
      dtype='object')

In [110]:
pay_df.columns

Index(['#JCAT', 'Piece', 'Name', 'LDate', 'TLast', 'TOp', 'TDate', 'TF',
       'Program', 'Plane', 'Att', 'Mvr', 'Class', 'Category', 'Result',
       'Control', 'Discipline', 'UNState', 'UNReg', 'UNPeriod', 'UNPerigee',
       'UNApogee', 'UNInc', 'DispEpoch', 'DispPeri', 'DispApo', 'DispInc',
       'Comment'],
      dtype='object')

In [111]:
orgs_df.columns

Index(['#Code', 'UCode', 'StateCode', 'Type', 'Class', 'TStart', 'TStop',
       'ShortName', 'Name', 'Location', 'Longitude', 'Latitude', 'Error',
       'Parent', 'ShortEName', 'EName', 'UName'],
      dtype='object')

In [112]:
lsite_df.columns

Index(['#Site', 'Code', 'UCode', 'Type', 'StateCode', 'TStart', 'TStop',
       'ShortName', 'Name', 'Location', 'Longitude', 'Latitude', 'Error',
       'Parent', 'ShortEName', 'EName', 'Group', 'UName'],
      dtype='object')

### Tidying up the datasets - remove columns that are not useful

In [113]:
# Remove columns we can't use
columns_to_keep = ['Piece', 'Type', 'Name', 'LDate', 'DDate', 'Status', 'Owner', 'State', 'Manufacturer', 'Mass', 'Length', 'Diameter', 'Span', 'Shape', 'Perigee', 'Apogee', 'Inc']
sdf = satcat_df[columns_to_keep]

# Rename Columns for easier reading
sdf.rename(columns={'Piece':'CosparID', 'LDate':'Launch Date', 'DDate':'Decay Date', 'Owner':'Operator Code','Manufacturer':'Manufacturer Code'}, inplace=True)

In [114]:
# Create the CombinedName column using owner_EName if available, otherwise owner_Name
orgs_df['CombinedName'] = orgs_df['EName'].fillna(orgs_df['Name'])

# Remove columns we can't use
orgs_columns_to_keep = ['#Code', 'ShortName', 'CombinedName', 'Longitude', 'Latitude', 'Parent']
odf = orgs_df[orgs_columns_to_keep]

# Rename Columns for easier reading
odf.rename(columns={'#Code':'Org Code', 'ShortName':'Short Name', 'CombinedName':'Long Name'}, inplace=True)

In [115]:

import time

# Set up the geocoder
geolocator = Nominatim(user_agent="geoapiExercises")
reverse = RateLimiter(geolocator.reverse, min_delay_seconds=1)

# Define a function to extract city and state
def get_location_info(lat, lon):
    time.sleep(1.1)
    print (".", end="", flush=True)
    try:
        location = reverse((lat, lon), language='en')
        if location and location.raw.get('address'):
            address = location.raw['address']
            city = address.get('city', address.get('town', address.get('village', '')))
            state = address.get('state', '')

            return f"{city}|{state}"
    except:
        return None
    
#filtered_odf = odf.tail(10)    

# Apply the function using a lambda
#filtered_odf['CityState'] = filtered_odf.apply(lambda row: get_location_info(row['Latitude'], row['Longitude']), axis=1)

### Combine the datasets

In [116]:
# Rename odf columns for Operator merge
owner_odf = odf.rename(columns=lambda x: f'Operator {x}' if x != 'Org Code' else 'Operator Code')
sdf = sdf.merge(owner_odf, on='Operator Code', how='left')

# Repeat for Manufacturer merge
manufacturer_odf = odf.rename(columns=lambda x: f'Manufacturer {x}' if x != 'Org Code' else 'Manufacturer Code')
sdf = sdf.merge(manufacturer_odf, on='Manufacturer Code', how='left')

In [117]:
sdf.columns

Index(['CosparID', 'Type', 'Name', 'Launch Date', 'Decay Date', 'Status',
       'Operator Code', 'State', 'Manufacturer Code', 'Mass', 'Length',
       'Diameter', 'Span', 'Shape', 'Perigee', 'Apogee', 'Inc',
       'Operator Short Name', 'Operator Long Name', 'Operator Longitude',
       'Operator Latitude', 'Operator Parent', 'Manufacturer Short Name',
       'Manufacturer Long Name', 'Manufacturer Longitude',
       'Manufacturer Latitude', 'Manufacturer Parent'],
      dtype='object')

In [118]:
payload_columns_to_keep = ['Piece', 'Program', 'Class', 'Category']
pdf = pay_df[payload_columns_to_keep]

# Rename 'Piece' to 'CosparID'
pdf.rename(columns={'Piece':'CosparID'}, inplace=True)

pdf.columns

Index(['CosparID', 'Program', 'Class', 'Category'], dtype='object')

In [119]:
sdf = sdf.merge(pdf, on='CosparID', how='left')

In [ ]:
# From the documentation:

class_json = {
  "A": "Academic / Amateur / Non-Profit",
  "B": "Commercial",
  "C": "Government (Civil)",
  "D": "Government (Defence / Military / Intelligence)",
}

type_json = {
  "P": "Payload",
  "C": "Component",
  "R": "Rocket Body",
  "D": "Debris",
  "S": "Suborbital payload",
  "X": "Deleted Object",
  "Z": "Spurious Entry"
}

category_json = {



}

status_json = {


}

In [ ]:
# Remove X and Z objects
sdf = sdf[~sdf['Type'].isin(['X', 'Z'])]
# Type field - Remove Sub-Types
sdf['Type'] = sdf['Type'].str[0]

# Map the descriptions
sdf['Class Desc'] = sdf['Class'].map(class_json)
sdf['Type Desc'] = sdf['Type'].map(type_json)

In [ ]:
sdf['Launch Date'] = sdf['Launch Date'].replace("-", np.nan)  # Shouldn't be any!
sdf = sdf.dropna(subset=['Launch Date'])
sdf['Decay Date'] = sdf['Decay Date'].replace("-", np.nan)

# Modify date columns
# Convert to datetime format
sdf['Launch Date'] = pd.to_datetime(sdf['Launch Date'], errors='coerce')
sdf['Decay Date'] = pd.to_datetime(sdf['Decay Date'], errors='coerce')

# Create a Year of Launch and Descent
sdf['Launch Year'] = sdf['Launch Date'].dt.year.astype('Int64')
sdf['Decay Year'] = sdf['Decay Date'].dt.year.astype('Int64')
sdf['Launch Month No'] = sdf['Launch Date'].dt.month.astype('Int64')
sdf['Decay Month No'] = sdf['Decay Date'].dt.month.astype('Int64')
sdf['Launch Month'] = sdf['Launch Date'].dt.strftime('%B')
sdf['Decay Month'] = sdf['Decay Date'].dt.strftime('%B')

# Replace null Decay Date with today's date
today = pd.Timestamp(datetime.today())
sdf['Effective Decay Date'] = sdf['Decay Date'].fillna(today)

# Calculate lifetime
delta = sdf['Effective Decay Date'] - sdf['Launch Date']
sdf['Lifespan Days'] = delta.dt.days
sdf['Lifespan Months'] = (delta.dt.days / 30.44).round(1)  # Approximate month length
sdf['Lifespan Years'] = (delta.dt.days / 365.25).round(2)  # Account for leap years

# Format as dd/mm/yyyy (because we are British)
sdf['Launch Date'] = sdf['Launch Date'].dt.strftime('%d/%m/%Y')
sdf['Decay Date'] = sdf['Decay Date'].dt.strftime('%d/%m/%Y')

# Drop Effective Decay Date column
sdf = sdf.drop('Effective Decay Date', axis=1)

Map country code to country name

In [123]:
# Convert alpha-2 to alpha-3
def alpha2_to_alpha3(code):
    try:
        return pycountry.countries.get(alpha_2=code).alpha_3
    except:
        return None

# Apply conversion to your State column
sdf['Country Code'] = sdf['State'].apply(alpha2_to_alpha3)
sdf['Country'] = sdf['Country Code'].apply(lambda x: pycountry.countries.get(alpha_3=x).name if x else None)

Status meanings:



In [124]:
sdf.Status.value_counts()

Status
R         32007
O         29905
OX         1169
L          1074
DK          465
DSO         458
N           308
AR          291
D           178
TFR          78
E            77
AO           65
GRP          61
ATT          46
DSA          44
C            23
ERR          22
R?           13
L?            2
REL           1
EVA DP        1
Name: count, dtype: int64

In [125]:
# Reorder alphabetically
sdf = sdf.reindex(sorted(sdf.columns), axis=1)